In [1]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from pandas.api.types import CategoricalDtype

from category_encoders import MEstimateEncoder
from category_encoders import CatBoostEncoder

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor


# Set Matplotlib defaults
plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

# Mute warnings
warnings.filterwarnings('ignore')

## Data Preprocessing ##

- **Load** the data from CSV files
- **Clean** the data to fix any errors or inconsistencies
- **Encode** the statistical data type (numeric, categorical)
- **Impute** any missing values

In [2]:
def load_data():
    # Read data
    data_dir = Path("/kaggle/input/datasets/joelzcharia/contraceptive-prevalence-survey/1987 Indonesia Contraception Prevalence Study.csv")
    df = pd.read_csv(data_dir)
    return df
    # Preprocessing
    # df = clean(df)
    # df = encode(df)
    # df = impute(df)

In [3]:
# Loading and Inspecting the Data

df = load_data()
print(df.shape, "\n")
print(df.dtypes, "\n")
print(df.head(10), "\n")
print(df.describe(), "\n")
print(df.info(), "\n")

(1473, 10) 

Age                          int64
Education                    int64
Partner Education            int64
Number of Children           int64
Religion = Islam             int64
Currently working            int64
Husband Occupation           int64
Standard of Living           int64
Media Exposure               int64
Contraceptive Method Used    int64
dtype: object 

   Age  Education  Partner Education   Number of Children  Religion = Islam  \
0   24          2                   3                   3                 1   
1   45          1                   3                  10                 1   
2   43          2                   3                   7                 1   
3   42          3                   2                   9                 1   
4   36          3                   3                   8                 1   
5   19          4                   4                   0                 1   
6   38          2                   3                   6           

In [4]:
# Check for Missing Data and Duplicated Data

df.isna().sum()
df.duplicated().sum()
df[df.duplicated()]

,Age,Education,Partner Education,Number of Children,Religion = Islam,Currently working,Husband Occupation,Standard of Living,Media Exposure,Contraceptive Method Used
79,38,4,4,1,1,0,1,4,0,1
167,26,4,4,1,1,1,1,4,0,1
224,47,4,4,4,1,1,1,4,0,1
270,30,4,4,2,1,1,1,4,0,1
299,26,4,4,1,1,1,1,4,0,1
394,29,4,4,0,1,0,2,4,0,1
414,20,2,3,3,1,1,3,4,0,1
462,36,4,4,3,1,1,1,4,0,2
492,37,4,4,3,1,1,1,4,0,2
528,29,4,4,2,1,0,1,3,0,2


In [5]:
# Recode Currently Working and Media Exposure to fix inverted coding
df["Currently working recoded"] = df["Currently working"].map({0:1, 1:0})
df["Media Exposure recoded"] = df["Media Exposure"].map({0:1, 1:0})

# Recode Contraceptive Method for more logical progression - 1 = none, 2 = short term, 3 = long term
df["Contraceptive Method Used recoded"] = df["Contraceptive Method Used"].map({1:1, 2:3, 3:2})
df[["Currently working recoded", "Media Exposure recoded", "Contraceptive Method Used recoded"]].head(10)



,Currently working recoded,Media Exposure recoded,Contraceptive Method Used recoded
0,0,1,1
1,0,1,1
2,0,1,1
3,0,1,1
4,0,1,1
5,0,1,1
6,0,1,1
7,1,1,1
8,0,1,1
9,0,0,1


In [6]:
# Create Age Groups

bins = [15, 19, 24, 29, 34, 39, 44, 49]
lab = ["15–19", "20–24", "25–29", "30–34", "35–39", "40–44", "45–49"]

df["Age Group"] = pd.cut(df["Age"], bins=bins, labels=lab, include_lowest=True)
df["Age Group"].value_counts()

Age Group
25–29    330
30–34    279
35–39    248
20–24    240
40–44    181
45–49    159
15–19     36
Name: count, dtype: int64

In [7]:
# Analyze percentage of women in each age group

age_distribution = (df['Age Group'].value_counts(normalize=True) * 100).round(2)

# pct_dist = df.groupby('Age Group').size()/len(df) * 100

print(age_distribution)

Age Group
25–29    22.40
30–34    18.94
35–39    16.84
20–24    16.29
40–44    12.29
45–49    10.79
15–19     2.44
Name: proportion, dtype: float64


In [8]:
# Analyze percentage of women in each age gourp using each contraceptive method

agecontra_dist = df.groupby('Age Group')['Contraceptive Method Used recoded'].value_counts(normalize=True).mul(100).round(2)

print(agecontra_dist)

Age Group  Contraceptive Method Used recoded
15–19      1                                    50.00
           2                                    41.67
           3                                     8.33
20–24      1                                    45.00
           2                                    40.83
           3                                    14.17
25–29      2                                    43.33
           1                                    38.79
           3                                    17.88
30–34      2                                    40.86
           1                                    34.77
           3                                    24.37
35–39      2                                    34.68
           1                                    34.27
           3                                    31.05
40–44      1                                    46.96
           3                                    33.15
           2                         

In [9]:
# Analyze percentage of women using each contraceptive method

contra_dist = df.groupby('Contraceptive Method Used recoded').size()/len(df) * 100

print(contra_dist)

Contraceptive Method Used recoded
1    42.701969
2    34.691107
3    22.606925
dtype: float64
